# 🌌 GigaGraph Unified Training Hub (v11.1.2)

Set `CONFIG['algorithm']` to switch between algorithms:
- `'aptp'` — APTP-GNN v8.12 (3.2B)
- `'noprop'` — NoProp Denoising v11.1 (3.2B, Sharded)

In [ ]:
# ── Cell 1: Config ──────────────────────────────────────────────────────────
CONFIG = {
    'algorithm': 'noprop',         # 'aptp' | 'noprop'
    'vocab_size': 128256,
    'lr': 1e-4,                    # Lowered for 3.2B stability
    'max_steps': 10000,            # Stop and verify after this many steps
    'warmup_steps': 400,
    'ignore_checkpoint': True,
    'wandb_project': 'gigagraph-v11',
    'wandb_run_id': 'noprop-scale-v11-1',
    # NoProp specific (3.2B Scale-up)
    'noprop_d_model': 3072,
    'noprop_depth': 32,
    'noprop_batch_size': 2,
    'noprop_seq_len': 1024,
}

In [ ]:
# ── Cell 2: Environment Setup & Aggressive Sync ─────────────────────────────
import os, sys, shutil, importlib, torch, wandb, time
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer

# Force Fresh Source (Nuclear Sync)
REPO_URL = 'https://github.com/ey3lock3r/gnn-llm.git'
if os.path.exists('gnn_llm'):
    shutil.rmtree('gnn_llm')
if os.path.exists('.git'):
    shutil.rmtree('.git')

os.system('git init . > /dev/null 2>&1')
os.system(f'git remote add origin {REPO_URL} > /dev/null 2>&1')
os.system('git fetch origin > /dev/null 2>&1 && git reset --hard origin/master > /dev/null 2>&1')

# Purge cached modules
for mod in list(sys.modules.keys()):
    if 'gnn_llm' in mod:
        del sys.modules[mod]

# Authorize W&B and HF from Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    os.environ['WANDB_API_KEY'] = user_secrets.get_secret("WANDB_API_KEY")
    os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")
    wandb.login(key=os.environ['WANDB_API_KEY'])
except Exception as e:
    print(f"⚠️ Secrets/W&B Login skipped: {e}")

# Install deps (Blazing fast via uv)
os.system('pip install -q uv > /dev/null 2>&1 && uv pip install --system -q datasets transformers python-dotenv wandb tqdm > /dev/null 2>&1')

import gnn_llm
importlib.reload(gnn_llm)
from gnn_llm import build_model
from gnn_llm.data.pipeline import GigaDataPipeline
from gnn_llm.training.utils import init_wandb
from gnn_llm.training.trainer import run_training

device = 'cuda' if torch.cuda.is_available() else 'cpu'
algo = CONFIG['algorithm']
print('Algorithm:', algo, '| Device:', device)

In [ ]:
# ── Cell 3: Build Model ──────────────────────────────────────────────────────
if algo == 'noprop':
    model = build_model(
        'noprop',
        vocab_size=CONFIG['vocab_size'],
        d_model=CONFIG['noprop_d_model'],
        depth=CONFIG['noprop_depth'],
        device=device,
        use_fp16=True
    )
    batch_size, seq_len = CONFIG['noprop_batch_size'], CONFIG['noprop_seq_len']
else:
    # APTP / Legacy fallback
    model = build_model(algo, vocab_size=CONFIG['vocab_size'], device=device)
    batch_size, seq_len = 2, 1024

print(f'Model: {algo} | Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
# ── Cell 4: Run Training ─────────────────────────────────────────────────────
timestamp = time.strftime("%m%d-%H%M%S")
unique_run_id = f"{CONFIG['wandb_run_id']}-{timestamp}"
init_wandb(CONFIG['wandb_project'], unique_run_id)

pipeline = GigaDataPipeline()
loader = pipeline.get_dataloader(batch_size=batch_size, seq_len=seq_len)

print(f'🚀 GigaGraph v11.1 Unified Trainer Launching [Run: {unique_run_id}]...')
run_training(model, loader, CONFIG)
wandb.finish()

In [ ]:
# ── Cell 5: 🗣️ Chat Verification ──────────────────────────────────────────────
print("\n--- 🗣️ Starting Chat Verification ---")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get('HF_TOKEN'))

prompts = [
    "The future of biologically plausible AI is",
    "The capital of France is",
    "Explain the theory of relativity in simple terms:"
]

model.eval()
for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    
    with torch.no_grad():
        # NoProp .generate() implements the diffusion denoising chain
        tokens = model.generate(inputs, max_new_tokens=30, temperature=0.8)
        
    response = tokenizer.decode(tokens[0], skip_special_tokens=True)
    print(f"GigaGraph: {response}")